# Construction material NLP — training and evaluation

**Goal:** Classify invoice descriptions into Concrete, Metals, Wood, Insulation and Glass using TF-IDF + Logistic Regression. Run cells in order.

**Evaluation design:** keep a frozen unseen-material-name challenge set, group-disjoint training/validation/test sets, select hyperparameters using validation only, and inspect final test results only after model selection. Synthetic-data metrics are **not** real-world performance estimates.

## 1. Imports and reproducibility

Select your project `.venv` kernel in VS Code. Install dependencies with `python -m pip install -e ".[notebook,dev]"` in the terminal.

In [ ]:
import json
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

SEED = 42
# Works when VS Code starts the kernel in notebooks/ or repository root.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/processed/construction_materials_clean.csv"
MODEL_PATH = ROOT / "models/material_classifier.joblib"
RESULTS_PATH = ROOT / "outputs/material_evaluation.json"
print("Project root:", ROOT)

## 2. Load and inspect the cleaned data

`description_clean` is the input feature **X**, and `category` is the target **y**. We keep `description` and `noise_type` for diagnostics.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run the EDA/cleaning notebook first.")
df = pd.read_csv(DATA_PATH)
required = {"description_clean", "category"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(df.columns))}")
print("Shape:", df.shape)
display(df.head(8))
display(df.isna().sum().rename("missing_values"))
display(df["category"].value_counts().rename("count"))

## 3. Remove unusable rows and label conflicts

This is deterministic record-level hygiene, not a learned transformation. Do not use global category frequencies or vocabulary to tune cleaning decisions after inspecting final tests.

In [ ]:
df = df.dropna(subset=["description_clean", "category"]).copy()
df["description_clean"] = df["description_clean"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()
df = df[(df["description_clean"] != "") & (df["category"] != "")]
df = df.drop_duplicates(subset=["description_clean", "category"])
conflict_counts = df.groupby("description_clean")["category"].nunique()
conflicting_texts = set(conflict_counts[conflict_counts > 1].index)
print("Conflicting descriptions removed:", len(conflicting_texts))
df = df[~df["description_clean"].isin(conflicting_texts)].copy()
labels = sorted(df["category"].unique())
if len(labels) < 2:
    raise ValueError("At least two classes are required")
print("Remaining rows:", len(df), "Labels:", labels)

## 4. Define unseen material names **before** fitting anything

These are *material names*, not new categories. We exclude all descriptions containing these expressions from training and validation. Review this fixed list against the generator **before** running final evaluation. If some patterns have no matches, the test still runs but has less coverage.

In [ ]:
UNSEEN_MATERIALS = {
    "Concrete": [r"\b(?:portland cement|cement mortar|betonmortel)\b"],
    "Metals": [r"\b(?:copper pipe|koperen buis|steel mesh)\b"],
    "Wood": [r"\b(?:mdf panel|hardhouten plank|oak flooring boards)\b"],
    "Insulation": [r"\b(?:acoustic insulation panel|dakisolatie)\b"],
    "Glass": [r"\b(?:float glass|gelaagd glas)\b"],
}

unseen_mask = pd.Series(False, index=df.index)
for category, patterns in UNSEEN_MATERIALS.items():
    for pattern in patterns:
        unseen_mask |= (df["category"].eq(category) &
                        df["description_clean"].str.contains(pattern, case=False, regex=True, na=False))
unseen_test = df[unseen_mask].copy()
known = df[~unseen_mask].copy()
print("Unseen-name test examples:", len(unseen_test))
display(unseen_test["category"].value_counts().reindex(labels, fill_value=0))

## 5. Group similar templates to reduce leakage

Replacing numbers/references creates a **heuristic** template identifier. GroupShuffleSplit keeps identical identifiers on one side of each split. It cannot detect all semantic near-duplicates; consider stronger grouping by source material name or invoice template for a production experiment.

In [ ]:
def template_group(text: str) -> str:
    text = text.casefold()
    text = re.sub(r"\b(?:po|ref|order)[-\s:]?\d+\b", "<ref>", text)
    text = re.sub(r"\d+(?:[.,/]\d+)*", "<num>", text)
    return re.sub(r"\s+", " ", text).strip()

known = known.copy()
# Ensure exact/heuristic templates from unseen material tests cannot enter training.
held_groups = set(unseen_test["description_clean"].map(template_group))
known = known[~known["description_clean"].map(template_group).isin(held_groups)].copy()

def group_split(frame, test_fraction, seed):
    groups = frame["description_clean"].map(template_group)
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
    idx_a, idx_b = next(splitter.split(frame, groups=groups))
    return frame.iloc[idx_a].copy(), frame.iloc[idx_b].copy()

train_val, known_test = group_split(known, 0.20, SEED)
train, validation = group_split(train_val, 0.20, SEED + 1)
assert set(train["category"]) == set(labels), "Some categories are missing in train; enlarge data."
for a, b in [(train, validation), (train, known_test), (validation, known_test)]:
    assert set(a["description_clean"].map(template_group)).isdisjoint(
        set(b["description_clean"].map(template_group)))
print({"train": len(train), "validation": len(validation),
       "known_test": len(known_test), "unseen_test": len(unseen_test)})

## 6. Create clean and noisy subsets of the held-out known-material test

`noise_type` describes **injected synthetic noise**, not necessarily noise remaining after cleaning. Thus these subsets measure the effect of the *original noise annotation on cleaned text*. For a true cleaning comparison, evaluate `description` versus `description_clean` on the **same** held-out rows.

In [ ]:
if "noise_type" in known_test.columns:
    noise = known_test["noise_type"].fillna("none").astype(str).str.strip().str.lower()
    clean_test = known_test[noise.isin(["none", "", "nan"])].copy()
    noisy_test = known_test[~noise.isin(["none", "", "nan"])].copy()
else:
    print("WARNING: noise_type not present. Clean/noisy breakdown unavailable.")
    clean_test = known_test.iloc[:0].copy()
    noisy_test = known_test.iloc[:0].copy()

split_sizes = pd.Series({"train": len(train), "validation": len(validation),
                         "clean test": len(clean_test), "noisy test": len(noisy_test),
                         "unseen names": len(unseen_test)})
display(split_sizes.to_frame("rows"))
ax = split_sizes.plot.bar(figsize=(8, 3), title="Split sizes")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()

## 7. Define TF-IDF + Logistic Regression

A scikit-learn `Pipeline` ensures TF-IDF vocabulary and IDF weights are learned **only from the training split** during `.fit()`. We compare a few predefined hyperparameters using validation macro F1; no test set is consulted.

In [ ]:
def build_pipeline(ngram_range=(1, 2), C=1.0):
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram_range,
                                  sublinear_tf=True, min_df=1)),
        ("classifier", LogisticRegression(C=C, max_iter=2000,
                                          class_weight="balanced", random_state=SEED)),
    ])

candidates = [
    {"ngram_range": (1, 2), "C": 1.0},
    {"ngram_range": (1, 2), "C": 3.0},
    {"ngram_range": (1, 3), "C": 1.0},
]
validation_results = []
for params in candidates:
    candidate = build_pipeline(**params)
    candidate.fit(train["description_clean"], train["category"])
    predicted = candidate.predict(validation["description_clean"])
    score = f1_score(validation["category"], predicted, average="macro", zero_division=0)
    validation_results.append({**params, "macro_f1": score})
display(pd.DataFrame(validation_results))
best_idx = max(range(len(validation_results)), key=lambda i: validation_results[i]["macro_f1"])
best_params = candidates[best_idx]
print("Selected on validation only:", best_params)

## 8. Refit the chosen pipeline on training + validation

Now that hyperparameters are fixed, we may use validation rows for fitting. The test sets remain unseen.

In [ ]:
final_train = pd.concat([train, validation], ignore_index=True)
model = build_pipeline(**best_params)
model.fit(final_train["description_clean"], final_train["category"])
print("TF-IDF vocabulary size:", len(model.named_steps["tfidf"].vocabulary_))
print("Final training rows:", len(final_train))

## 9. Final evaluation: clean, noisy, unseen names

**Only run this section once the design and hyperparameters are frozen.** Report sample counts: small or empty subsets cannot support strong conclusions.

In [ ]:
def evaluate_set(frame, name):
    if frame.empty:
        print(f"{name}: EMPTY — no score reported")
        return {"n": 0, "accuracy": None, "macro_f1": None, "weighted_f1": None}
    y_true = frame["category"]
    y_pred = model.predict(frame["description_clean"])
    print(f"\n{name} — n={len(frame)}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    return {
        "n": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
    }

results = {
    "clean": evaluate_set(clean_test, "Clean known-material test"),
    "noisy": evaluate_set(noisy_test, "Noisy known-material test"),
    "unseen": evaluate_set(unseen_test, "Previously unseen material names"),
}
display(pd.DataFrame(results).T)

## 10. Confusion matrices and error analysis

Inspect which categories are confused and whether errors are caused by ambiguous terms, abbreviations or unseen names. Do **not** change the model after seeing final-test errors and report the same test as an unbiased score.

In [ ]:
for name, frame in [("Clean", clean_test), ("Noisy", noisy_test), ("Unseen names", unseen_test)]:
    if frame.empty:
        continue
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(frame["category"],
        model.predict(frame["description_clean"]), labels=labels,
        xticks_rotation=45, ax=ax, colorbar=False)
    ax.set_title(f"{name} test — confusion matrix")
    plt.tight_layout()
    plt.show()

In [ ]:
error_frames = []
for name, frame in [("clean", clean_test), ("noisy", noisy_test), ("unseen", unseen_test)]:
    if frame.empty:
        continue
    temp = frame.copy()
    temp["prediction"] = model.predict(temp["description_clean"])
    temp["test_set"] = name
    error_frames.append(temp[temp["prediction"] != temp["category"]])
if error_frames:
    errors = pd.concat(error_frames, ignore_index=True)
    print("Total misclassified examples:", len(errors))
    display(errors[[c for c in ["test_set", "description", "description_clean", "category", "prediction", "noise_type"] if c in errors.columns]].head(20))

## 11. Try a few new descriptions

This is a demonstration, not a substitute for held-out evaluation. Logistic Regression probabilities are not guaranteed to be calibrated.

In [ ]:
examples = [
    "C30/37 betonmortel 12 m3 levering",
    "galv steel beam HEA200 qty 12",
    "spruce timber 45x145mm",
    "PIR isolatieplaat 100mm",
    "HR++ dubbel glas 4-15-4",
]
predictions = model.predict(examples)
probabilities = model.predict_proba(examples).max(axis=1)
display(pd.DataFrame({"description": examples, "prediction": predictions,
                      "max_model_probability": probabilities.round(3)}))

## 12. Save model and evaluation metrics

The `models/` and `outputs/` directories may be ignored by Git. Keep the notebook and dataset generator versioned; record package versions or use a lockfile for reproducibility.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
payload = {
    "selected_params": {"ngram_range": list(best_params["ngram_range"]), "C": best_params["C"]},
    "validation_macro_f1": validation_results[best_idx]["macro_f1"],
    "test_metrics": results,
    "warning": "Synthetic-data results are not real-world performance estimates",
}
RESULTS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH)
print("Saved metrics:", RESULTS_PATH)

## 13. Interpretation and next experiments

1. Compare macro F1 across clean, noisy and unseen-material-name test sets, **including sample sizes**.
2. Compare word TF-IDF with character n-grams (`analyzer="char_wb", ngram_range=(3, 5)`) using a **new validation-only experiment**.
3. For a fair raw-versus-cleaned comparison, build matched held-out inputs from `description` and `description_clean`, while keeping the training procedure fixed.
4. For a defensible final result, collect independently labeled real invoices and freeze them before any model decisions.

**Leakage warning:** the template grouping is heuristic; identical material names can still appear in different groups. A stronger benchmark holds out source material families or original invoice templates, not just numeric variants.